### Backup 1: Applying hard constrains

For some problems, it is advantageous to apply prior knowledge about the solution in the network architecture, instead of the training process via additional loss terms. For example, in simple domains a Dirichlet boundary condition could be added to the network output with a corresponding characteristic function. Since the boundary condition is then naturally fulfilled, one has to consider fewer terms in the final loss and the optimization may become easier.

Here we consider the example: 
\begin{align*}
    \partial_y u(x,y) &= \frac{u(x,y)}{y}, \text{ in } (0, 1) \times (0, 1) \\
    u_1(x, 0) &= 0 , \text{ for } x \in (0, 1) \\
    u_2(x, 1) &= \sin(20\pi*x) , \text{ for } x \in (0, 1) \\
    \vec{n} \nabla u(x, y) &= 0 , \text{ for } x \in \{0, 1\}, y \in (0, 1)\\
\end{align*}

This problem has the simple solution $u(x, y) = y\sin(20\pi x)$. But because of the high frequencies of the sinus term, a training of the boundary condition is rather problematic. One reason for this is the usage of the MSE inside the loss function, which promotes to just learn the mean value of our boundary function. Another reason is the spectral bias of neural networks.

In the following, the problem is firstly implemented the normal way, where all above equations are used to define different loss terms. This will not lead to a correct/useful solution. (Until the first results, no comments will be given regarding the implementation. Since just the basics are used)

In [ ]:
# # This block is for GPU selection. Please execute.
# import pathlib
# import os
# user = int(str(pathlib.Path().resolve())[22:24])
# os.environ["CUDA_VISIBLE_DEVICES"]= str(user % 4)

In [ ]:
import qewton
import math

# Parameter of the problem
width, height = 1.0, 1.0
frequenz = 20.0 * math.pi

cos_node = qewton.bb.Cos()
def bc_fn(x):
    return cos_node(frequenz*x[:, 0:1])

In [ ]:
# First we define the space and domain:
X = qewton.Variable("x", dim=2)
U = qewton.Variable("u", dim=1)

square = qewton.geometries.Rectangle(X, [0.0, 0.0], 1.0, 1.0)

In [ ]:
# TODO: Create the samplers for the boundary and inside the domain. The boundary samplers, 
# should sample either for the left and right boundary(x_sampler), or for the top and bottom (y_sampler)
inner_sampler = qewton.RandomUniformSampler(square, n_points=50000)

def filter_left_right(x: X):
    return (x[:, 0] == 0.0) | (x[:, 0] == 1.0)

def filter_top_bottom(x: X):
    return (x[:, 1] == 0.0) | (x[:, 1] == 1.0)

boundary_x = qewton.RandomUniformSampler(square.boundary, n_points=20000, filter_fn=filter_left_right)
boundary_y = qewton.RandomUniformSampler(square.boundary, n_points=20000, filter_fn=filter_top_bottom)

In [ ]:
model = qewton.FCN(in_neurons=X, out_neurons=U, hidden_neurons=32, n_hidden_layers=4)

First we try to learn the solution without any extensions of the generall PINN approach, e.g. we just use all
conditions for training.

In [ ]:
tol = 0.0001 # to not divide by small numbers if y is close to 0

def pde_residual(u: U, x : X):
    y = x[:, 1:2]
    # TODO implement the PDE
    return u.gradient(x)[:, 1:2] - u/(y + tol)

pde_pipeline = qewton.PINNPipeline(sampler=inner_sampler, models=[model], residual=pde_residual)
pde_constraint = pde_pipeline.constraint

In [ ]:
def dirichlet_residual(u : U, x : X):
    return u - x[:, 1:2] * bc_fn(x)

# TODO: add the PINNCondition for the Dirichlet boundary
dirichlet_pipeline = qewton.PINNPipeline(sampler=boundary_y, 
                                         models=[model], 
                                         residual=dirichlet_residual, 
                                         residual_name='dirichlet_constraint')
dirichlet_constraint = dirichlet_pipeline.constraint

In [ ]:
def neumann_residual(u : U, x: X):
    return u.gradient(x)[:, 0:1]

neumann_pipeline = qewton.PINNPipeline(sampler=boundary_x, 
                                        models=[model], 
                                        residual=neumann_residual, 
                                        residual_name='neumann_constraint', 
                                        weight=1.0/60.0)
neumann_constraint = neumann_pipeline.constraint

In [ ]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=5000,
    lr_scheduler=qewton.optim.StepLR(step_size=1000, gamma=0.5)
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase],
    graphs=[pde_pipeline, neumann_pipeline, dirichlet_pipeline],
    training_objectives=[pde_constraint, neumann_constraint, dirichlet_constraint],
    device=qewton.cuda(0),
)

trainer.run()

The training was done with Adam and a learning rate of 0.001 and just for 5000 iterations. But just choosing a different algorithm or lr values will not improve the results easily. The loss will in general hover around some constant value near 0.25.

Having a look at the learned solution, most of the time we either get a solution that is close to zero or has just captured a few oscillations of the boundary function.

In [ ]:
layout = pde_pipeline.visualize(model.output_ports[0], plot_type=qewton.viz.SurfacePlot, z=U)
qewton.viz.Figure(layout).show()

We can also plot the error, since we know the analytical solution.

In [ ]:
import numpy as np
def reference_solution(points):
    x, y = points[:, 0:1], points[:, 1:2]
    return np.cos(20*math.pi*x)*y

layout = pde_pipeline.visualize(
    model.output_ports[0],
    reference=reference_solution,
    plot_type=qewton.viz.SurfacePlot,
    z=U,
    max_vertex_distance={X: 0.05},
)
qewton.viz.Figure(layout, 
                  renderer=qewton.viz.renderers.PlotlyRenderer(theme=qewton.viz.Theme(plot_width=250, plot_height=300))).show()

Now, we want to build the boundary condition into out network.
We start by resetting our network (defining a new one). With fewer parameters, because we implement the high frequencies via hard constrains and therefore the network now has to learn a simpler function.

For the hard constraints, we create the following python-function, where we choose the ansatz to just multiply the network output with the boundary function:

In [ ]:
from typing import Annotated

def tensor_of(var):
    return Annotated[qewton.TensorType, qewton.DataConfiguration(qewton.EllipsisAxes(), qewton.FeatureAxes(var))]

In [ ]:
model = qewton.FCN(in_neurons=X, out_neurons=U, hidden_neurons=10, n_hidden_layers=2)

def constrain_fn(x : tensor_of(X)) -> tensor_of(U):
    u = model(x)
    return u*bc_fn(x)

constrained_model = qewton.FromFunctionNode(constrain_fn, backend=qewton.DEFAULT_DL_BACKEND)

The hard constraints now have to be always be applied inside our residual functions of the different conditions. One important point is, that our constraints do not automatically fulfill any boundary condition. So we still have all previous conditions.

Choosing for example the constraint:
, would naturally fulfill the boundary condition at . What we choose is somewhat arbitrary, important for this problem is that the oscillation appears in the constrain function.

In [ ]:
def pde_residual(u: U, x : X):
    y = x[:, 1:2]
    return u.gradient(x)[:, 1:2] - u/(y + tol)

pde_pipeline = qewton.PINNPipeline(sampler=inner_sampler, models=[constrained_model], residual=pde_residual)
pde_constraint = pde_pipeline.constraint

# TODO: Implement Dirichlet and Neumann residual
def dirichlet_residual(u : U, x : X):
    return u - x[:, 1:2] * bc_fn(x)

dirichlet_pipeline = qewton.PINNPipeline(sampler=boundary_y, 
                                         models=[constrained_model], 
                                         residual=dirichlet_residual, 
                                         residual_name='dirichlet_constraint')
dirichlet_constraint = dirichlet_pipeline.constraint

def neumann_residual(u : U, x: X):
    return u.gradient(x)[:, 0:1]

neumann_pipeline = qewton.PINNPipeline(sampler=boundary_x, 
                                        models=[constrained_model], 
                                        residual=neumann_residual, 
                                        residual_name='neumann_constraint',
                                        weight=1.0/60.0)
neumann_constraint = neumann_pipeline.constraint

In [ ]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=5000,
    lr_scheduler=qewton.optim.StepLR(step_size=1000, gamma=0.5)
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase],
    graphs=[pde_pipeline, neumann_pipeline, dirichlet_pipeline],
    training_objectives=[pde_constraint, neumann_constraint, dirichlet_constraint],
    device=qewton.cuda(0),
)

trainer.run()

In [ ]:
layout = pde_pipeline.visualize(constrained_model.output_ports[0], plot_type=qewton.viz.SurfacePlot, z=U)
qewton.viz.Figure(layout).show()

In [ ]:
import numpy as np
def reference_solution(points):
    x, y = points[:, 0:1], points[:, 1:2]
    return np.cos(20*math.pi*x)*y

layout = pde_pipeline.visualize(
    constrained_model.output_ports[0],
    reference=reference_solution,
    plot_type=qewton.viz.SurfacePlot,
    z=U,
    max_vertex_distance={X: 0.05},
)
qewton.viz.Figure(layout, 
                  renderer=qewton.viz.renderers.PlotlyRenderer(theme=qewton.viz.Theme(plot_width=250, plot_height=300))).show()